# Pipeline Evaluation: MobileNetV4 → PGA-UNet

Đánh giá pipeline 2 tầng trên cùng tập test.

## Cách tính metric kết hợp

| Trường hợp | GT | Classifier | Segmenter chạy? | Score |
|---|---|---|---|---|
| TP + Seg tốt | Có bệnh | Có bệnh ✅ | Có | Dice thực tế |
| FN (bỏ sót) | Có bệnh | Không bệnh ❌ | Không | **Dice = 0** |
| TN (đúng) | Không bệnh | Không bệnh ✅ | Không | Không tính vào Dice |
| FP (báo nhầm) | Không bệnh | Có bệnh ❌ | Có | **Dice = 0** (GT rỗng) |

### Công thức
```
Pipeline_Dice = (1/N_pos) × Σ [Dice_seg_i × I(cls_i == CÓ_BỆNH)]
              ≈ Sensitivity_cls × Mean_Dice_seg
```

### 3 phần đánh giá
- **Phần 1**: Classification đơn thuần (BTXRD_FS, có + không bệnh) → Sensitivity, Specificity, AUC
- **Phần 2**: Segmentation đơn thuần (dataset_BTXRD/test, N ảnh dương) → Dice, IoU, HD95, CBL
- **Phần 3**: Pipeline kết hợp (chạy CLS trước rồi SEG) → Pipeline_Dice, Pipeline_HD95

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
%cd /content
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

import os

# Clone PGA-UNet source
if not os.path.exists('Prompt-Guided-XRay-Segmentation'):
    !git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

# Install
!pip install -q timm gdown tqdm opencv-python scipy openpyxl

# Download segmentation dataset (positive cases + polygon annotations)
!gdown 1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW -O dataset_BTXRD.zip -q
!unzip -oq dataset_BTXRD.zip
!rsync -a dataset_BTXRD/ Prompt-Guided-XRay-Segmentation/dataset_BTXRD/

# Download classification dataset (positive + negative cases)
!gdown 1vv3OZBZ5QS4-RzFkkrdsMz3UpAMjIplY -O BTXRD_FS.zip -q
!unzip -oq BTXRD_FS.zip

print('\n✅ Setup xong')

In [ ]:
# ── Cell 2: Download checkpoints ─────────────────────────────────────────────
import gdown, os

# ← Điền Drive ID của best_mobilenetv4.pth vào đây
CLS_CKPT_ID = 'YOUR_MOBILENETV4_DRIVE_ID'
PGA_CKPT_ID = '1qVb-PlJrSk1R3MPBrbB8qll7N8FqcDx0'

os.makedirs('/content/checkpoints', exist_ok=True)
os.makedirs('/content/Prompt-Guided-XRay-Segmentation/checkpoints', exist_ok=True)

cls_ckpt = '/content/checkpoints/best_mobilenetv4.pth'
pga_ckpt = '/content/Prompt-Guided-XRay-Segmentation/checkpoints/pga_unet_expB_best.pth'

if not os.path.exists(cls_ckpt):
    gdown.download(f'https://drive.google.com/uc?id={CLS_CKPT_ID}', cls_ckpt, quiet=False)
if not os.path.exists(pga_ckpt):
    gdown.download(f'https://drive.google.com/uc?id={PGA_CKPT_ID}', pga_ckpt, quiet=False)

assert os.path.exists(cls_ckpt), f'❌ Không tìm thấy: {cls_ckpt}'
assert os.path.exists(pga_ckpt), f'❌ Không tìm thấy: {pga_ckpt}'
print('✅ Checkpoints sẵn sàng')

In [ ]:
# ── Cell 3: Load models ───────────────────────────────────────────────────────
import sys, torch, torch.nn as nn, timm
sys.path.insert(0, '/content/Prompt-Guided-XRay-Segmentation')
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLS_THRESHOLD = 0.3   # threshold dùng nhất quán với MobileNetV4_BTXRD_dataset.ipynb
SEG_IMG_SIZE  = 512   # PGA-UNet
CLS_IMG_SIZE  = 256   # MobileNetV4 (resize trước khi đưa vào)

# --- MobileNetV4 ---
cls_model = timm.create_model(
    'mobilenetv4_hybrid_medium.ix_e550_r384_in1k',
    pretrained=False, num_classes=0  # tắt head gốc, thay thủ công
)
cls_model = timm.create_model('mobilenetv4_hybrid_medium.ix_e550_r384_in1k', pretrained=False)
cls_model.classifier = nn.Linear(cls_model.classifier.in_features, 1)
cls_model.load_state_dict(torch.load(cls_ckpt, map_location=DEVICE))
cls_model = cls_model.to(DEVICE).eval()
print('✅ MobileNetV4 loaded')

# --- PGA-UNet ---
seg_model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
seg_model.load_state_dict(torch.load(pga_ckpt, map_location=DEVICE, weights_only=True))
seg_model.eval()
print('✅ PGA-UNet loaded')

In [ ]:
# ── Cell 4: Helpers ───────────────────────────────────────────────────────────
import cv2, json, numpy as np
from PIL import Image
from torchvision import transforms
from scipy.ndimage import binary_erosion, distance_transform_edt

# --- Preprocessing ---

# MobileNetV4: grayscale → replicate 3 kênh, resize 256, ImageNet normalize
cls_transform = transforms.Compose([
    transforms.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def preprocess_cls(img_path):
    img = Image.open(img_path).convert('RGB')  # X-quang grayscale → RGB (replicate 3 kênh)
    return cls_transform(img).unsqueeze(0).to(DEVICE)  # (1,3,256,256)

# PGA-UNet: grayscale, resize 512, normalize [-1, 1]
seg_transform = transforms.Compose([
    transforms.Resize((SEG_IMG_SIZE, SEG_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def preprocess_seg(img_path):
    img = Image.open(img_path).convert('L')   # grayscale
    return seg_transform(img).unsqueeze(0).to(DEVICE)  # (1,1,512,512)

# --- Prompt: Gaussian heatmap từ polygon annotation (zoom_out) ---

def make_heatmap_from_json(json_path, img_path, S=512, zoom_ratio=0.30):
    """
    Đọc tất cả polygon trong JSON → tạo Gaussian heatmap (plateau-style).
    Giống dataset.py create_plateau_heatmap, zoom_out 30%.
    """
    orig = cv2.imread(img_path)
    oh, ow = orig.shape[:2]
    sx, sy = S / ow, S / oh

    hm = np.zeros((S, S), dtype=np.float32)

    with open(json_path) as f:
        ann = json.load(f)

    for obj in ann.get('objects', []):
        pts = np.array(obj['points']['exterior'], dtype=np.float32)
        xs, ys = pts[:, 0] * sx, pts[:, 1] * sy
        x1, x2 = xs.min(), xs.max()
        y1, y2 = ys.min(), ys.max()
        bw, bh = x2 - x1, y2 - y1

        bx1 = max(0, int(x1 - bw * zoom_ratio))
        by1 = max(0, int(y1 - bh * zoom_ratio))
        bx2 = min(S, int(x2 + bw * zoom_ratio))
        by2 = min(S, int(y2 + bh * zoom_ratio))

        roi = np.zeros((S, S), dtype=np.float32)
        if bx2 > bx1 and by2 > by1:
            roi[by1:by2, bx1:bx2] = 1.0
            roi = cv2.GaussianBlur(roi, (31, 31), 0)
        hm = np.maximum(hm, roi)

    return torch.from_numpy(hm).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,512,512)

# --- Metrics ---

def extract_lcc(m):
    if m.sum() == 0: return m
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    return m if n <= 1 else (lbl == (1 + np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_hd95(pred, gt, img_size=SEG_IMG_SIZE):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(img_size)  # prediction rỗng → worst case
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    return float(img_size) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred, gt):
    if gt.sum() == 0: return None
    if pred.sum() == 0: return 0.0
    ys, xs = np.where(gt > 0.5); yp, xp = np.where(pred > 0.5)
    diag = np.sqrt((ys.max() - ys.min())**2 + (xs.max() - xs.min())**2) + 1e-6
    d = np.sqrt((xp.mean() - xs.mean())**2 + (yp.mean() - ys.mean())**2)
    return float(np.clip(1.0 - d / diag, 0.0, 1.0))

def calc_seg_metrics(pred_prob, gt_mask):
    """Tính 6 metrics từ prob map + GT mask. Trả về dict."""
    pm = extract_lcc((pred_prob > 0.5).astype(np.float32))
    gm = (gt_mask > 0.5).astype(np.float32)
    eps = 1e-6
    tp = (pm * gm).sum()
    fp = (pm * (1 - gm)).sum()
    fn = ((1 - pm) * gm).sum()
    return dict(
        dice  = float((2*tp + eps) / (2*tp + fp + fn + eps)),
        iou   = float((tp + eps)   / (tp + fp + fn + eps)),
        pre   = float((tp + eps)   / (tp + fp + eps)),
        rec   = float((tp + eps)   / (tp + fn + eps)),
        hd95  = calc_hd95(pm, gm),
        cbl   = calc_cbl(pm, gm) or 0.0,
    )

def gt_mask_from_json(json_path, img_path, S=512):
    """Tạo GT binary mask từ tất cả polygon trong file JSON annotation."""
    orig = cv2.imread(img_path)
    oh, ow = orig.shape[:2]
    sx, sy = S / ow, S / oh
    mask = np.zeros((S, S), dtype=np.float32)

    with open(json_path) as f:
        ann = json.load(f)

    for obj in ann.get('objects', []):
        pts = np.array(obj['points']['exterior'], dtype=np.float32)
        pts[:, 0] *= sx; pts[:, 1] *= sy
        cv2.fillPoly(mask, [pts.astype(np.int32)], 1.0)

    return mask

print('✅ Helpers sẵn sàng')

## Phần 1 — Đánh giá phân loại (BTXRD_FS)
Chạy MobileNetV4 trên tập test của BTXRD_FS (có cả ảnh bình thường + ảnh bệnh).

In [ ]:
# ── Cell 5: Phần 1 — Classification ──────────────────────────────────────────
import pandas as pd, numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             roc_auc_score, classification_report)

DATA_DIR   = '/content/BTXRD_FS'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
EXCEL_PATH = os.path.join(DATA_DIR, 'dataset.xlsx')

df = pd.read_excel(EXCEL_PATH)
df['image_id'] = df['image_id'].str.replace('.jpeg', '.png', regex=False)
df['image_id'] = df['image_id'].str.replace('.jpg',  '.png', regex=False)

# Dùng cùng split với MobileNetV4_BTXRD_dataset.ipynb (random_state=42)
_, val_df   = train_test_split(df,    test_size=0.3,  random_state=42, stratify=df['tumor'])
val_df, test_df = train_test_split(val_df, test_size=0.5, random_state=42, stratify=val_df['tumor'])
test_df = test_df.reset_index(drop=True)

print(f'Test set: {len(test_df)} ảnh  '
      f'(có bệnh: {test_df["tumor"].sum()}, bình thường: {(test_df["tumor"]==0).sum()})')

# --- Chạy inference ---
cls_probs, cls_labels = [], []
cls_model.eval()

with torch.no_grad():
    for _, row in test_df.iterrows():
        img_path = os.path.join(IMAGES_DIR, row['image_id'])
        if not os.path.exists(img_path):
            continue
        inp  = preprocess_cls(img_path)
        prob = torch.sigmoid(cls_model(inp)).item()
        cls_probs.append(prob)
        cls_labels.append(int(row['tumor']))

cls_probs  = np.array(cls_probs)
cls_labels = np.array(cls_labels)
cls_preds  = (cls_probs >= CLS_THRESHOLD).astype(int)

# --- Metrics ---
cm = confusion_matrix(cls_labels, cls_preds)
tn, fp, fn, tp = cm.ravel()

sensitivity  = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # Recall dương tính
specificity  = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # Recall âm tính
accuracy     = accuracy_score(cls_labels, cls_preds)
auc          = roc_auc_score(cls_labels, cls_probs)

print('\n' + '='*50)
print('  KẾT QUẢ PHÂN LOẠI (MobileNetV4)')
print('='*50)
print(f'  Threshold         : {CLS_THRESHOLD}')
print(f'  Accuracy          : {accuracy:.4f}')
print(f'  Sensitivity (Rec+): {sensitivity:.4f}  ← % ảnh bệnh phát hiện đúng')
print(f'  Specificity (Rec-): {specificity:.4f}  ← % ảnh bình thường loại đúng')
print(f'  AUC-ROC           : {auc:.4f}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')

# Lưu để dùng ở Phần 3
CLS_SENSITIVITY = sensitivity

## Phần 2 — Đánh giá phân đoạn thuần (dataset_BTXRD/test)
Chỉ chạy PGA-UNet, không qua bước classification. Đây là số liệu gốc (Đóng góp 1).

In [ ]:
# ── Cell 6: Phần 2 — Segmentation thuần ──────────────────────────────────────
from tqdm import tqdm

IMG_DIR  = '/content/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/test/images'
JSON_DIR = '/content/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/test/annotations'

seg_only_results = []

json_files = sorted([f for f in os.listdir(JSON_DIR) if f.endswith('.json')])
print(f'Test samples: {len(json_files)}')

seg_model.eval()
with torch.no_grad():
    for jf in tqdm(json_files, desc='Seg-only'):
        base     = os.path.splitext(jf)[0]
        img_path = os.path.join(IMG_DIR,  base + '.png')
        jsn_path = os.path.join(JSON_DIR, jf)

        if not os.path.exists(img_path):
            continue

        img_t    = preprocess_seg(img_path)           # (1,1,512,512)
        prompt_t = make_heatmap_from_json(jsn_path, img_path)  # (1,1,512,512)
        gt_mask  = gt_mask_from_json(jsn_path, img_path)        # (512,512)

        prob_np  = torch.sigmoid(seg_model(img_t, prompt_t))[0, 0].cpu().numpy()
        metrics  = calc_seg_metrics(prob_np, gt_mask)
        metrics['sample'] = base
        seg_only_results.append(metrics)

keys = ['dice', 'iou', 'pre', 'rec', 'hd95', 'cbl']
print('\n' + '='*50)
print('  KẾT QUẢ PHÂN ĐOẠN THUẦN (PGA-UNet, zoom_out)')
print('='*50)
for k in keys:
    vals = [r[k] for r in seg_only_results]
    arrow = '↑' if k != 'hd95' else '↓'
    print(f'  {k.upper():<6} {arrow}: {np.mean(vals):.4f}  (N={len(vals)})')

## Phần 3 — Đánh giá pipeline kết hợp
Với mỗi ảnh dương tính trong test set:
1. Chạy MobileNetV4 → có bệnh / không bệnh
2. Nếu phát hiện có bệnh → chạy PGA-UNet → lấy Dice thực tế
3. Nếu bỏ sót (FN) → Dice = 0, HD95 = IMG_SIZE (penalty tối đa)

**Pipeline_Dice ≈ Sensitivity × Segmentation_Dice**

In [ ]:
# ── Cell 7: Phần 3 — Pipeline kết hợp ────────────────────────────────────────

pipeline_results = []
n_detected  = 0  # TP: MobileNetV4 nói CÓ bệnh (đúng)
n_missed    = 0  # FN: MobileNetV4 nói KHÔNG bệnh (sai, bỏ sót)

cls_model.eval()
seg_model.eval()

with torch.no_grad():
    for jf in tqdm(json_files, desc='Pipeline'):
        base     = os.path.splitext(jf)[0]
        img_path = os.path.join(IMG_DIR,  base + '.png')
        jsn_path = os.path.join(JSON_DIR, jf)

        if not os.path.exists(img_path):
            continue

        gt_mask = gt_mask_from_json(jsn_path, img_path)

        # ── Tầng 1: Classification ──
        cls_inp  = preprocess_cls(img_path)
        cls_prob = torch.sigmoid(cls_model(cls_inp)).item()
        has_disease = cls_prob >= CLS_THRESHOLD

        if has_disease:
            # ── Tầng 2: Segmentation ──
            n_detected += 1
            img_t    = preprocess_seg(img_path)
            prompt_t = make_heatmap_from_json(jsn_path, img_path)
            prob_np  = torch.sigmoid(seg_model(img_t, prompt_t))[0, 0].cpu().numpy()
            metrics  = calc_seg_metrics(prob_np, gt_mask)
        else:
            # FN: bỏ sót → penalty tối đa
            n_missed += 1
            metrics  = dict(dice=0.0, iou=0.0, pre=0.0, rec=0.0,
                            hd95=float(SEG_IMG_SIZE), cbl=0.0)

        metrics['sample']      = base
        metrics['cls_prob']    = cls_prob
        metrics['cls_correct'] = has_disease  # tất cả GT đều positive
        pipeline_results.append(metrics)

# --- Tổng hợp kết quả ---
N_total    = len(pipeline_results)
detect_rate = n_detected / N_total if N_total > 0 else 0.0

print('\n' + '='*60)
print('  KẾT QUẢ PIPELINE KẾT HỢP (MobileNetV4 → PGA-UNet)')
print('='*60)
print(f'  Tổng ảnh dương tính : {N_total}')
print(f'  Phát hiện đúng (TP) : {n_detected}  ({detect_rate*100:.1f}%)')
print(f'  Bỏ sót (FN)        : {n_missed}   ({(1-detect_rate)*100:.1f}%)')
print()

labels = {
    'dice': ('DICE',  '↑'), 'iou':  ('IoU',   '↑'),
    'pre':  ('Pre',   '↑'), 'rec':  ('Rec',   '↑'),
    'hd95': ('HD95',  '↓'), 'cbl':  ('CBL',   '↑'),
}
print(f'  {"Metric":<8} {"Arrow":>2}  {"Pipeline":>10}  {"Seg-only":>10}  {"Lý thuyết":>12}')
print('  ' + '-'*52)

seg_means = {k: np.mean([r[k] for r in seg_only_results]) for k in labels}

for k, (name, arrow) in labels.items():
    pipe_mean = np.mean([r[k] for r in pipeline_results])
    seg_mean  = seg_means[k]
    # Lý thuyết: sensitivity × seg_mean (chỉ đúng cho Dice/IoU/Pre/Rec/CBL)
    theory = detect_rate * seg_mean if k != 'hd95' else seg_mean + (1 - detect_rate) * SEG_IMG_SIZE
    print(f'  {name:<8} {arrow:>2}  {pipe_mean:>10.4f}  {seg_mean:>10.4f}  {theory:>12.4f}')

print()
print(f'  Pipeline_Dice = Sensitivity × Seg_Dice')
print(f'                = {detect_rate:.4f} × {seg_means["dice"]:.4f}')
print(f'                = {detect_rate * seg_means["dice"]:.4f}')
print(f'  (Thực nghiệm)   {np.mean([r["dice"] for r in pipeline_results]):.4f}')

In [ ]:
# ── Cell 8: Bảng tổng hợp cuối + lưu CSV ──────────────────────────────────────
import csv, matplotlib.pyplot as plt

# --- Lưu CSV pipeline ---
os.makedirs('results', exist_ok=True)
csv_path = 'results/pipeline_combined_results.csv'
fieldnames = ['sample', 'cls_prob', 'cls_correct', 'dice', 'iou', 'pre', 'rec', 'hd95', 'cbl']
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in pipeline_results:
        w.writerow({k: r.get(k, '') for k in fieldnames})
print(f'✅ CSV saved: {csv_path}')

# --- Bảng 3 tầng in ra màn hình ---
print('\n' + '='*70)
print('  BẢNG TỔNG HỢP 3 TẦNG ĐÁNH GIÁ')
print('='*70)
print(f'  {"Tầng":<35} {"Metric chính":<25} {"Giá trị":>8}')
print('  ' + '-'*65)
print(f'  {"1. Phân loại (MobileNetV4)":<35} {"Sensitivity":>25} {"{:.4f}".format(CLS_SENSITIVITY):>8}')
print(f'  {"":<35} {"Specificity":>25} {"[từ Phần 1]":>8}')
print(f'  {"":<35} {"AUC-ROC":>25} {"[từ Phần 1]":>8}')
print(f'  {"2. Phân đoạn thuần (PGA-UNet)":<35} {"Dice (zoom_out)":>25} {"{:.4f}".format(seg_means["dice"]):>8}')
print(f'  {"":<35} {"HD95 (px, zoom_out)":>25} {"{:.2f}".format(seg_means["hd95"]):>8}')
print(f'  {"3. Pipeline kết hợp":<35} {"Pipeline_Dice":>25} {"{:.4f}".format(np.mean([r["dice"] for r in pipeline_results])):>8}')
print(f'  {"":<35} {"Pipeline_HD95 (px)":>25} {"{:.2f}".format(np.mean([r["hd95"] for r in pipeline_results])):>8}')
print(f'  {"":<35} {"Detection rate":>25} {"{:.4f}".format(detect_rate):>8}')
print('='*70)

# --- Biểu đồ so sánh ---
metrics_show  = ['dice', 'iou', 'pre', 'rec', 'cbl']
seg_vals      = [seg_means[k] for k in metrics_show]
pipe_vals     = [np.mean([r[k] for r in pipeline_results]) for k in metrics_show]
theory_vals   = [detect_rate * seg_means[k] for k in metrics_show]

x = np.arange(len(metrics_show))
w = 0.28
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, seg_vals,    w, label='Seg-only (PGA)', color='steelblue')
ax.bar(x,     pipe_vals,   w, label='Pipeline thực nghiệm', color='tomato')
ax.bar(x + w, theory_vals, w, label=f'Lý thuyết (Sens={CLS_SENSITIVITY:.2f} × Seg)', color='gold', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([k.upper() for k in metrics_show])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score'); ax.set_title('So sánh Seg-only vs Pipeline kết hợp')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/pipeline_combined_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved: results/pipeline_combined_chart.png')